In [ ]:
!pip install -U transformers datasets peft bitsandbytes accelerate
!pip install pandas openpyxl bert-score sentencepiece

In [2]:
import pandas as pd
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model

In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
df = pd.read_excel("")
df = df.head(30000)

In [5]:
mild_df = df[df["Severity"] == "Mild"]
moderate_df = df[df["Severity"] == "Moderate"]
critical_df = df[df["Severity"] == "Critical"]

In [6]:
def convert(df):
    return Dataset.from_pandas(df[["Question","Answer"]])

mild_ds = convert(mild_df)
moderate_ds = convert(moderate_df)
critical_ds = convert(critical_df)

In [ ]:
from huggingface_hub import login
login('')

In [ ]:
model_name = "aubmindlab/aragpt2-base"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

In [ ]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["c_attn", "c_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [10]:
max_input = 256
max_output = 256

def preprocess(example):

    prompt = f"""السؤال: {example["Question"]}
الاجابة: {example["Answer"]}"""

    tokens = tokenizer(
        prompt,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    labels = tokens["input_ids"].copy()

    # ignore padding in loss
    labels = [-100 if token == tokenizer.pad_token_id else token for token in labels]

    tokens["labels"] = labels

    return tokens

In [ ]:
mild_ds = mild_ds.map(preprocess)
moderate_ds = moderate_ds.map(preprocess)
critical_ds = critical_ds.map(preprocess)

In [12]:
from datasets import concatenate_datasets

phase1 = mild_ds

phase2 = concatenate_datasets([
    mild_ds,
    moderate_ds
])

phase3 = concatenate_datasets([
    mild_ds,
    moderate_ds,
    critical_ds
])

In [ ]:
def train_phase(dataset, model, lr, output_dir):

    args = TrainingArguments(
      output_dir = output_dir,
      per_device_train_batch_size = 1,
      gradient_accumulation_steps = 8,
      learning_rate = lr,
      num_train_epochs = 3,
      fp16=True,
      logging_steps = 50,
      save_strategy = "steps",
      save_steps = 500,
      report_to = "none",
      remove_unused_columns=False
   )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=dataset
    )

    trainer.train()
    # trainer.train(
    #     resume_from_checkpoint=""
    # )

    return trainer.model

In [ ]:
# Phase 1
model = train_phase(
    phase1,
    model,
    lr=2e-5,
    output_dir=""
)

In [ ]:
# Phase 2
model = train_phase(
    phase2,
    model,
    lr=1e-5,
    output_dir=""
)

In [ ]:
# Phase 3
model = train_phase(
    phase3,
    model,
    lr=8e-6,
    output_dir=""
)

In [ ]:
model.save_pretrained("")
tokenizer.save_pretrained("")

In [ ]:
def build_prompt(row):
    return f"""{row['Question']}"""

def generate_answer(row):
    prompt = build_prompt(row)


    inputs = tokenizer(prompt, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

In [ ]:
import pandas as pd
from bert_score import score
from tqdm.auto import tqdm # Import tqdm

# Load evaluation set
eval_df = pd.read_excel("")
output_path = ""


# Optional: limit rows for testing
eval_df = eval_df.head(2000)

preds = []
refs = []
questions = []

# Generate responses
for i, (_, row) in enumerate(tqdm(eval_df.iterrows(), total=len(eval_df)), start=1):
    pred = generate_answer(row)
    pred = pred.split(row["Question"], 1)[-1].strip()
    pred = pred.split("الاجابة:", 1)[-1].strip()

    preds.append(pred)
    refs.append(row["Answer"])
    questions.append(row["Question"])

    # 🔹 Save every 100 rows
    if i % 100 == 0:
        output_df = pd.DataFrame({
            "Question": questions,
            "Answer": refs,
            "Model_Response": preds
        })
        output_df.to_excel(output_path, index=False)


# Save Question, Answer, and Model Response
output_df = pd.DataFrame({
    "Question": questions,
    "Answer": refs,
    "Model_Response": preds
})
output_df.to_excel(output_path, index=False)

# Compute overall BERTScore (dataset-level)
P, R, F1 = score(
    preds,
    refs,
    lang="ar",
    verbose=True
)

overall_f1 = F1.mean().item()
print("Overall BERTScore F1:", overall_f1)